In [1]:
import os
from google.adk.agents import Agent
from google.adk.models.lite_llm import LiteLlm
from google.adk.sessions import InMemorySessionService
from google.adk.runners import Runner
from google.genai import types 
from typing import Optional, Dict, Any

import warnings
warnings.filterwarnings("ignore")

import logging
logging.basicConfig(level=logging.CRITICAL)


import litellm
import logging

logging.getLogger("LiteLLM").setLevel(logging.CRITICAL)
logging.getLogger("litellm").setLevel(logging.CRITICAL)
logging.disable(logging.CRITICAL)  

In [2]:
MODEL_GPT = "groq/openai/gpt-oss-120b"
llm = LiteLlm(model=MODEL_GPT, reasoning_format="hidden")

print(
    llm.llm_client.completion(
        model=llm.model,
        messages=[
            {
                "role": "user",
                "content": "Are you ready?"
            }
        ],
        tools=[]
    )
)
print("\nGroq is ready for use.")

ModelResponse(id='chatcmpl-b8e7d16c-e8ee-471c-8371-149d8be7f033', created=1789550848, model='openai/gpt-oss-120b', object='chat.completion', system_fingerprint='fp_5082008e34', choices=[Choices(finish_reason='stop', index=0, message=Message(content="Absolutely—I'm ready! How can I help you today?", role='assistant', tool_calls=None, function_call=None, provider_specific_fields=None, reasoning='User asks "Are you sure?" Actually "Are you ready?" It\'s a simple question. The assistant should respond. There\'s no policy violation. Just respond positively.'))], usage=Usage(completion_tokens=54, prompt_tokens=75, total_tokens=129, completion_tokens_details=CompletionTokensDetailsWrapper(accepted_prediction_tokens=None, audio_tokens=None, reasoning_tokens=33, rejected_prediction_tokens=None, text_tokens=None, image_tokens=None, video_tokens=None), prompt_tokens_details=None, queue_time=0.368020672, prompt_time=0.003096006, completion_time=0.112419172, total_time=0.115515178), usage_breakdown

In [10]:
from neo4j_for_adk import graphdb

In [3]:
from helper import make_agent_caller

In [4]:
def say_hello(person_name: str) -> dict:
    """Formats a welcome message to a named person. 

    Args:
        person_name (str): the name of the person saying hello

    Returns:
        dict: A dictionary containing the results of the query.
              Includes a 'status' key ('success' or 'error').
              If 'success', includes a 'query_result' key with an array of result rows.
              If 'error', includes an 'error_message' key.
    """
    return graphdb.send_query("RETURN 'Hello to you, ' + $person_name AS reply",
    {
        "person_name": person_name
    })

In [5]:
def say_goodbye() -> dict:
    """Provides a simple farewell message to conclude the conversation."""
    return graphdb.send_query("RETURN 'Goodbye from Cypher!' as farewell")

# Sub-Agents (Greeting & Farewell)

In [6]:
# Greeting Agent 
greeting_subagent = Agent(
    model=llm,
    name="greeting_subagent_v1",
    instruction="You are the Greeting Agent. Your ONLY task is to provide a friendly greeting to the user. "
                "Use the 'say_hello' tool to generate the greeting. "
                "If the user provides their name, make sure to pass it to the tool. "
                "Do not engage in any other conversation or tasks.",
    description="Handles simple greetings and hellos using the 'say_hello' tool.", 
    tools=[say_hello],
)
print(f"Agent '{greeting_subagent.name}' created.")


Agent 'greeting_subagent_v1' created.


In [7]:
# Farewell Agent 
farewell_subagent = Agent(
    model=llm, 
    name="farewell_subagent_v1",
    instruction="You are the Farewell Agent. Your ONLY task is to provide a polite goodbye message. "
                "Use the 'say_goodbye' tool when the user indicates they are leaving or ending the conversation "
                "(e.g., using words like 'bye', 'goodbye', 'thanks bye', 'see you'). "
                "Do not perform any other actions.",
    description="Handles simple farewells and goodbyes using the 'say_goodbye' tool.", 
    tools=[say_goodbye],
)
print(f"Agent '{farewell_subagent.name}' created.")

Agent 'farewell_subagent_v1' created.


# Root Agent with Sub-Agents

In [8]:
root_agent = Agent(
    name="friendly_agent_team_v1", 
    model=llm,
    description="The main coordinator agent. Delegates greetings/farewells to specialists.",
    instruction="""You are the main Agent coordinating a team. Your primary responsibility is to be friendly.
 
                You have specialized sub-agents: 
                1. 'greeting_agent': Handles simple greetings like 'Hi', 'Hello'. Delegate to it for these. 
                2. 'farewell_agent': Handles simple farewells like 'Bye', 'See you'. Delegate to it for these. 

                Analyze the user's query. If it's a greeting, delegate to 'greeting_agent'. 
                If it's a farewell, delegate to 'farewell_agent'. 
                
                For anything else, respond appropriately or state you cannot handle it.
                """,
    tools=[], 
    sub_agents=[greeting_subagent, farewell_subagent]
)


print(f"Root Agent '{root_agent.name}' created with sub-agents: {[sa.name for sa in root_agent.sub_agents]}")


Root Agent 'friendly_agent_team_v1' created with sub-agents: ['greeting_subagent_v1', 'farewell_subagent_v1']


# Interact with the Agent Team

In [11]:
from helper import make_agent_caller

root_agent_caller = await make_agent_caller(root_agent)

async def run_team_conversation():
    await root_agent_caller.call("Hello I'm ABK", True)

    await root_agent_caller.call("Thanks, bye!", True)
await run_team_conversation()



>>> User Query: Hello I'm ABK
  [Event] Author: friendly_agent_team_v1, Type: Event, Final: False, Content: parts=[Part(
  function_call=FunctionCall(
    args={
      'agent_name': 'greeting_subagent_v1'
    },
    id='fc_ee844bb0-ddb2-4338-80f4-c4b04eb686f7',
    name='transfer_to_agent'
  )
)] role='model'
  [Event] Author: friendly_agent_team_v1, Type: Event, Final: False, Content: parts=[Part(
  function_response=FunctionResponse(
    id='fc_ee844bb0-ddb2-4338-80f4-c4b04eb686f7',
    name='transfer_to_agent',
    response={
      'result': None
    }
  )
)] role='user'
  [Event] Author: greeting_subagent_v1, Type: Event, Final: False, Content: parts=[Part(
  function_call=FunctionCall(
    args={
      'person_name': 'ABK'
    },
    id='fc_5e45f065-16b6-4315-a545-83104c1483f2',
    name='say_hello'
  )
)] role='model'
  [Event] Author: greeting_subagent_v1, Type: Event, Final: False, Content: parts=[Part(
  function_response=FunctionResponse(
    id='fc_5e45f065-16b6-4315-a545-8

# Memory and Personalization with session state

State-Aware hello/goodbye Tool

In [12]:
from google.adk.tools.tool_context import ToolContext

def say_hello_stateful(user_name:str, tool_context:ToolContext):
    """Says hello to the user, recording their name into state.
    
    Args:
        user_name (str): The name of the user.
    """
    tool_context.state["user_name"] = user_name
    print("\ntool_context.state['user_name']:", tool_context.state["user_name"])
    return graphdb.send_query(
        f"RETURN 'Hello to you, ' + $user_name + '.' AS reply",
    {
        "user_name": user_name
    })

In [13]:
def say_goodbye_stateful(tool_context: ToolContext) -> dict:
    """Says goodbye to the user, reading their name from state."""
    user_name = tool_context.state.get("user_name", "stranger")
    print("\ntool_context.state['user_name']:", user_name)
    return graphdb.send_query("RETURN 'Goodbye, ' + $user_name + ', nice to chat with you!' AS reply",
    {
        "user_name": user_name
    })


print("State-aware 'say_hello_stateful' and 'say_goodbye_stateful' tools defined.")


State-aware 'say_hello_stateful' and 'say_goodbye_stateful' tools defined.


# Redefine Sub-Agents and Update Root Agent

In [14]:
greeting_agent_stateful = Agent(
    model=llm,
    name="greeting_agent_stateful_v1",
    instruction="You are the Greeting Agent. Your ONLY task is to provide a friendly greeting using the 'say_hello' tool. Do nothing else.",
    description="Handles simple greetings and hellos using the 'say_hello_stateful' tool.",
    tools=[say_hello_stateful],
)
print(f" Agent '{greeting_agent_stateful.name}' redefined.")


 Agent 'greeting_agent_stateful_v1' redefined.


In [17]:
farewell_agent_stateful = Agent(
    model=llm,
    name="farewell_agent_stateful_v1",
    instruction="You are the Farewell Agent. Your ONLY task is to provide a polite goodbye message using the 'say_goodbye_stateful' tool. Do not perform any other actions.",
    description="Handles simple farewells and goodbyes using the 'say_goodbye_stateful' tool.",
    tools=[say_goodbye_stateful],
)
print(f" Agent '{farewell_agent_stateful.name}' redefined.")

 Agent 'farewell_agent_stateful_v1' redefined.


In [18]:
root_agent_stateful = Agent(
    name="friendly_team_stateful", 
    model=llm,
    description="The main coordinator agent. Delegates greetings/farewells to specialists.",
    instruction="""You are the main Agent coordinating a team. Your primary responsibility is to be friendly.

                You have specialized sub-agents: 
                1. 'greeting_agent_stateful': Handles simple greetings like 'Hi', 'Hello'. Delegate to it for these. 
                2. 'farewell_agent_stateful': Handles simple farewells like 'Bye', 'See you'. Delegate to it for these. 

                Analyze the user's query. If it's a greeting, delegate to 'greeting_agent_stateful'. If it's a farewell, delegate to 'farewell_agent_stateful'. 
                
                For anything else, respond appropriately or state you cannot handle it.
                """,
        tools=[], 
        sub_agents=[greeting_agent_stateful, farewell_agent_stateful], 
    )

print(f"Root Agent '{root_agent_stateful.name}' created using agents with stateful tools.")


Root Agent 'friendly_team_stateful' created using agents with stateful tools.


In [19]:
root_stateful_caller = await make_agent_caller(root_agent_stateful)

session = await root_stateful_caller.get_session()

print(f"Initial State: {session.state}")

Initial State: {}


In [ ]:
async def run_stateful_conversation():
    await root_stateful_caller.call("Hello, I'm ABK!")

    await root_stateful_caller.call("Thanks, bye!")

# Execute the conversation using await in an async context (like Colab/Jupyter)
await run_stateful_conversation()

session = await root_stateful_caller.get_session()

print(f"\nFinal State: {session.state}")

In [ ]:
async def run_interactive_conversation():
    while True:
        user_query = input("Ask me something (or type 'exit' to quit): ")
        if user_query.lower() == 'exit':
            break
        response = await root_stateful_caller.call(user_query)
        print(f"Response: {response}")

# Execute the interactive conversation
await run_interactive_conversation()